# A real `rclpy` node, running in this notebook

This is genuinely compiled `rclpy` — the same ROS 2 rolling Python client library used on real robots — running via [`xeus-python`](https://github.com/jupyter-xeus/xeus-python) on [`jupyterlite-xeus`](https://github.com/jupyter-xeus/jupyterlite-xeus), talking to ROS 2 over [`rmw_zenoh_pico`](https://github.com/esol-community/rmw_zenoh_pico). Not a mock, not a REST call dressed up to look like `rclpy` — the exact same `rclpy`/`rmw_zenoh_pico` build used by the [standalone rclpy talker demo](../index.html#demos), just running inside a notebook kernel instead of a plain web page, so you can edit and re-run cells to try things out.

**You need a zenoh router reachable at `ws://127.0.0.1:7447` for anything below to actually publish/receive** — see the main page's setup box for the one-line `pixi exec` command. Without one, `rclpy.init()`/node creation still work, but publishing will fail (matches the standalone demos' behavior, [documented limitation](../index.html#limits): the connect address is compiled in, not runtime-configurable, so this notebook can't point at a different router either).

In [ ]:
import os

# Same workarounds the standalone rclpy demo needs (see ../browser_demo/rclpy_boot.c
# and talker_rclpy.py) -- rcl_logging's default backend isn't wired up on this
# platform, and rmw_zenoh_pico is the only RMW built into this environment anyway.
os.environ['RCL_LOGGING_IMPLEMENTATION'] = 'rcl_logging_noop'
os.environ['RMW_IMPLEMENTATION'] = 'rmw_zenoh_pico'

import asyncio
import rclpy
from rclpy.node import Node
from rclpy.event_handler import PublisherEventCallbacks, SubscriptionEventCallbacks
from std_msgs.msg import String

rclpy.init(args=[])

# rmw_zenoh_pico's z_open() kicks off the WebSocket connection to the zenoh
# router asynchronously and can't block waiting for it -- this build has
# neither pthreads nor Asyncify, so nothing can synchronously block until the
# socket's "open" event fires. That means the *first* Node() construction
# attempt right after rclpy.init() is *expected* to fail every time,
# regardless of how fast the router responds: the WebSocket can only
# progress once this call yields back to the browser's event loop, which a
# plain synchronous call never does. Every other demo in this project
# (talker_rclc.c, talker_rclpy.py, ...) works around this with a retry loop
# driven from outside Python (JS ticking a tick() function); a notebook
# cell has no such external driver, but xeus-python's own asyncio
# integration genuinely pumps the browser event loop across `await
# asyncio.sleep()` (confirmed via ../scratch/jupyterlite_env_test/
# browser_test.html), so retrying here with a real await does the same job
# in a single cell.
async def _init_node_with_retry(max_attempts=30, delay=0.3):
    for attempt in range(1, max_attempts + 1):
        try:
            return Node('wasm_rclpy_jupyter', enable_rosout=False, start_parameter_services=False)
        except Exception as e:
            print(f'Node() attempt {attempt} not ready yet ({e!r}), retrying...')
            await asyncio.sleep(delay)
    raise RuntimeError(f'Node() never succeeded after {max_attempts} attempts')

# enable_rosout=False, start_parameter_services=False: sidesteps two more
# rclpy features that would otherwise pull in the same typesupport gap this
# whole pipeline works around elsewhere -- see ../docs/demo_env.md.
node = await _init_node_with_retry()
print('Node created:', node.get_name())

## Publisher and subscriber

Both on `chatter_jupyter` — publish from here and you'll see it echoed back below, or open [the rclc talker demo](../demos/index_rclc.html) or [the rclpy talker demo](../index.html#demos) in another tab and watch its own topic independently, or point a native `ros2 topic echo /chatter_jupyter` at the same router to see this notebook's own messages arrive outside the browser entirely.

`use_default_callbacks=False`: `rmw_zenoh_pico` doesn't support QoS event callbacks yet (the same [known limitation](../index.html#limits) the compiled demos work around) — leaving the defaults on raises instead of silently no-oping.

In [ ]:
received = []

def _on_message(msg):
    received.append(msg.data)
    print('received:', msg.data)

pub = node.create_publisher(
    String, 'chatter_jupyter', 10,
    event_callbacks=PublisherEventCallbacks(use_default_callbacks=False))

sub = node.create_subscription(
    String, 'chatter_jupyter', _on_message, 10,
    event_callbacks=SubscriptionEventCallbacks(use_default_callbacks=False))

print('Publisher + subscriber ready on chatter_jupyter')

## Play around

Edit the string below and re-run this cell as many times as you like — each run publishes one message, then spins the node briefly so any reply (including your own, echoed back by the subscriber above) has a chance to arrive before the cell finishes.

In [ ]:
msg = String()
msg.data = 'hello from the notebook!'
pub.publish(msg)
print('published:', msg.data)

for _ in range(20):
    rclpy.spin_once(node, timeout_sec=0.1)

## Cleanup

Run this once you're done. Unlike a normal `rclpy` script, this only destroys the node -- it deliberately skips `rclpy.shutdown()` (see the code cell's own comment for why). That means the zenoh session this kernel opened is never released, so re-running the setup cell above in this *same* kernel session won't work; restart the kernel (Kernel menu, or the restart button above) if you want to run through the notebook again from scratch.

In [ ]:
node.destroy_node()

# Deliberately not calling rclpy.shutdown() here: it tears down the shared
# zenoh session (rmw_shutdown() -> rmw_context_fini() ->
# zenoh_pico_destroy_session()), which crashes outright with an uncaught
# "RuntimeError: memory access out of bounds" wasm trap in this build --
# reproduced even with no publisher/subscriber ever created, right after a
# plain destroy_node(), so it's not something this demo's own pub/sub usage
# triggers. A real fix needs matching zenoh-pico/rmw_zenoh_pico source to
# the exact pinned build and instrumenting zenoh_pico_destroy_session()
# (the likeliest suspects are the z_drop() calls that tear down the
# WebSocket transport) -- out of scope for this notebook. Since a wasm trap
# is a native crash, not a Python exception, no try/except here can catch
# or work around it either; skipping the call entirely is the only
# option short of that deeper fix. destroy_node() alone already cleans up
# this node's own publisher/subscriber/entities without touching the
# session, so nothing above is left dangling.
print('node destroyed (zenoh session left open deliberately -- see comment above)')